# Machine Learning --- Jour 7
## Hugging Face Transformers: tweets politiques camerounais

** Master GI3 -- Approche Probleme / Solution**

On veut classer automatiquement des tweets politiques camerounais (positif / negatif / neutre) -- sans ecrire de regles a la main.

---

## 0. Mise en place

1. Installer les bibliotheques si besoin : 'pip install transformers torch'
2. Placez 'tweets_cameroun.csv' dans le meme dossier que ce notebook.
3. La premiere execution du modele Hugging Face telechargera 1 Go.
** Une connexion internet est indispensable pour la premiere fois.**

In [11]:
import os
if os.path.exists("tweets_cameroun.csv"):
   print("fichier trouve")
else:
   print("Fichier introuvable : verifiez son emplacement")

fichier trouve


## 1. Explorer le dataset

In [12]:
import pandas as pd

df=pd.read_csv("tweets_cameroun.csv")
print(f"Total tweets: {len(df)}")
print()
print("Repartition des sentiments:")
print(df["sentiment"].value_counts())
print()
print("Exemples de tweets:")
print(df.groupby("sentiment").head(1)[["tweet","sentiment"]].to_string())

Total tweets: 45

Repartition des sentiments:
sentiment
positif    15
negatif    15
neutre     15
Name: count, dtype: int64

Exemples de tweets:
                                                                                      tweet sentiment
0    Le nouveau pont de Yaoundé va enfin désengorger la circulation. Bravo au gouvernement!   positif
15  La corruption gangrène toujours nos institutions. Rien ne change vraiment dans ce pays.   negatif
30             Le Parlement camerounais se réunit aujourd'hui pour examiner le budget 2026.    neutre


## 2. Premiere tentative : regle naive par mots-cles

In [13]:
mots_positifs=["bravo","bien","bon","progres","fier","amelior"]
mots_negatifs=["corruption","honte","manque","insupport","souffr","echec"]

def regle_naive(tweet):

    tweet_lower=tweet.lower()
    score_pos= sum( 1 for m in mots_negatifs if m in tweet_lower)
    score_neg= sum( 1 for m in mots_negatifs if m in tweet_lower)

    if score_pos> score_neg: return "positif"
    elif score_neg > score_pos: return "negatif"
    else:  return "neutre"

df["pred_naive"]= df['tweet'].apply(regle_naive)
acc_naive= (df["pred_naive"]== df["sentiment"]).mean()
print(f"Accuracy regle naive : {acc_naive:.3f}")
print()

# Exemples d'erreurs

erreurs_naive= df[df["pred_naive"]!= df["sentiment"]].head(5)
for _,row in erreurs_naive.iterrows():
    print(f"Tweet: {row['tweet'][:60]}......")
    print(f" Reel: {row['sentiment']} | Predit : {row['pred_naive']}")
    print()

Accuracy regle naive : 0.333

Tweet: Le nouveau pont de Yaoundé va enfin désengorger la circulati......
 Reel: positif | Predit : neutre

Tweet: Fier du Cameroun! Notre équipe nationale a brillé. Vive les ......
 Reel: positif | Predit : neutre

Tweet: L'électrification de mon village est en cours. On voit enfin......
 Reel: positif | Predit : neutre

Tweet: Les routes de Douala commencent à s'améliorer. Continuez dan......
 Reel: positif | Predit : neutre

Tweet: Bonne nouvelle: le taux d'inscription scolaire au Cameroun a......
 Reel: positif | Predit : neutre



## 3. Qu'est ce qu'un Transformer ?

Un Transformer comprend **le contexte** complet d'une phrase:
- "Ce projet est **vraiment bien**" -> positif
- "Ce projet n'est **pas bien**" ->negatif (meme mot "bien", sens oppose)

**Hugging** heberge des milliers de modeles pre-entraines sur des milliards de textes.
utilisables en 3 lignes de code.

## 4. Pipeline Hugging Face -- sentiment multilingue

In [18]:
from transformers import pipeline
import warnings; warnings.filterwarnings('ignore')

print("Chargement du modele(peut prendre 1-2 minutes la 1ere fois)...")

# Modele multiligne entraine sur 198 millions de tweets en 100 langues (dont francais)
clf= pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

print("Modele charge!")

# Test rapide
tweet_test= "La corruption gangrene toujours nos institutions. Rien ne charge dans ce pays."
resultat = clf(tweet_test)
print(f"\n Tweet: {tweet_test}")
print(f"Resultat : {resultat}")

Chargement du modele(peut prendre 1-2 minutes la 1ere fois)...


KeyboardInterrupt: 

In [ ]:
# Traduction des labels anglais vers le francais
label_fr={"Positive": "positif", "Negative":"negatif", "Neutral":"neutre"}

# Appliquer sur tous les tweets (batch_size=8 : 8 tweets a la fois)
print("Analyse de tous les tweets en cours....")
resultats= clf(df["tweet"].tolist(), batch_size=8)

df["pred_hf"]=[labels_fr.get(r["label", r["label"]])]